In [1]:
import os
import sys
import json
import time
import kaggle
from kagglehub.competition import competition_download

import numpy as np
import pandas as pd

os.environ['KAGGLE_USERNAME'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['username']
os.environ['KAGGLE_KEY'] = json.load(open('/home/osman/.config/kaggle/kaggle.json'))['key']
path = competition_download('ing-hubs-turkiye-datathon')

/home/osman-tekdamar/Projects/forCV/ING_Datathon/.claude/worktrees/init-claude-md/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FileNotFoundError: [Errno 2] No such file or directory: '/home/osman/.config/kaggle/kaggle.json'

In [ ]:
customer_history = pd.read_csv(f"{path}/customer_history.csv")
customers = pd.read_csv(f"{path}/customers.csv")
referance_data = pd.read_csv(f"{path}/referance_data.csv")
referance_data_test = pd.read_csv(f"{path}/referance_data_test.csv")
sample_submission = pd.read_csv(f"{path}/sample_submission.csv") 

In [ ]:
referance_data["ref_date"] = pd.to_datetime(referance_data["ref_date"])
referance_data_test["ref_date"] = pd.to_datetime(referance_data_test["ref_date"])
customer_history["date"] = pd.to_datetime(customer_history["date"])

In [ ]:
customer_history[customer_history["cust_id"] == 125]

In [ ]:
customer_history[customer_history["cust_id"] == int(customers.loc[customers["tenure"] == customers["tenure"].max(), "cust_id"])]

In [ ]:
customer_history.groupby("cust_id").min()["date"].unique()

In [ ]:
customer_history.groupby("cust_id").max()["date"].min() , customer_history.groupby("cust_id").min()["date"].max()

In [ ]:
referance_data["ref_date"].min() , referance_data["ref_date"].max()

In [ ]:
referance_data_test["ref_date"].min(), referance_data_test["ref_date"].max()

In [ ]:
#büyülü kodlar
def diff_month(date1, date2):    
    return round((date1 - date2).days/30)

def format_date(date_series):
    # Pandas Series/Timestamp nesnesini YY-MM formatına çevirir
    return date_series.strftime('%y-%m')

# Hesaplamalar
first_customer_date = customer_history.groupby("cust_id").min()["date"].max()
train_start_date = referance_data["ref_date"].min()
train_end_date = referance_data["ref_date"].max()
test_start_date = referance_data_test["ref_date"].min()
test_end_date = referance_data_test["ref_date"].max()

# Formatted dates
first_customer_formatted = format_date(first_customer_date)
train_start_formatted = format_date(train_start_date)
train_end_formatted = format_date(train_end_date)
test_start_formatted = format_date(test_start_date)
test_end_formatted = format_date(test_end_date)

# Ay farkları
months_to_train_start = diff_month(train_start_date, first_customer_date)
months_train_period = diff_month(train_end_date, train_start_date)
months_to_test_start = diff_month(test_start_date, first_customer_date)
months_test_period = diff_month(test_end_date, test_start_date)

# Boşluk hesaplamaları
train_col1_spaces = ' ' * (months_to_train_start - len(first_customer_formatted))
train_col2_spaces = '  ' * (months_train_period - (len(train_start_formatted) + len(train_end_formatted) + 1))
test_col1_spaces = ' ' * (months_to_test_start - len(first_customer_formatted))
test_col2_spaces = ' ' * (months_test_period - (len(test_start_formatted) + len(test_end_formatted) + 1))

# Çizgi uzunlukları
train_line1 = '-' * months_to_train_start
train_line2 = '-' * months_train_period
test_line1 = '-' * months_to_test_start
test_line2 = '-' * months_test_period

# Print
print(f"""
   {' '*10} Veri Seti Zaman Çizelgesi {' '*10} \n
         |{train_line1}|{train_line2}|
Eğitim {first_customer_formatted}{train_col1_spaces} {train_start_formatted}{train_col2_spaces} {train_end_formatted} \n
         |{test_line1}|{test_line2}|
Test   {first_customer_formatted}{test_col1_spaces} {test_start_formatted}{test_col2_spaces} {test_end_formatted} 
""")

In [ ]:
referance_data_test

In [ ]:
customer_history[customer_history["cust_id"] == 58923]

In [ ]:
customers.loc[customers["cust_id"] == 58923]

In [ ]:
referance_data[referance_data["cust_id"] == 58923]

In [ ]:
customers[customers["cust_id"] == 143454]

In [ ]:
referance_data_test["cust_id"].value_counts().sort_values(ascending=False)

In [ ]:
referance_data["cust_id"].value_counts().sort_values(ascending=False)

farklı zaman damgalararında aynı müşteriye ait hiç gözlem yok hem train hem test setinde bu zaman bağımsız çalışmamızı rahatlatır

In [ ]:
customers.head()

In [ ]:
set(referance_data_test["cust_id"]) - set(customer_history["cust_id"]) 

In [ ]:
len(set(customer_history["cust_id"]) - set(referance_data_test["cust_id"]) )

test setinde olup history de olmayan yok yani yeni müşterinin gelmediğini varsayıyoruz
history de olan ama test de olmaya 133287 gözlem var

In [ ]:
customers.isna().sum()

work_sector de nan değerler var neden kaynaklandığını inceleyelim

In [ ]:
customers[customers["work_sector"].isna()]["work_type"].value_counts()

açıkca belli ki sektörü olmayan kişiler aslında çalışmadıkları için sektörü yok yanlış girdi sözkonusu değil bu durumda doğrudan work_type ile dolduracağım

In [ ]:
customers["work_sector"] = customers["work_sector"].fillna(customers["work_type"])

In [ ]:
customers

In [ ]:
customers["cust_age_month"] = (customers["age"] * 12)  - customers["tenure"]

In [ ]:
customers[customers["cust_age_month"] / 12 < 18]

bankacılıkta müşteri olabilmek için 18 üstü olman gerek bu durumda 18 altında üye olmuş gibi gözüken satırları kaldırmalıyız ama belkide başka ülkelerde bu durum normaldir

In [ ]:
customers[customers["work_type"] == "Retired"]["age"].describe()

In [ ]:
customers["age"].describe()

In [ ]:
customers.groupby("religion").agg({"age":"median"})

In [ ]:
customers["age"].min()

In [ ]:
customer_history.isna().sum()

In [ ]:
customer_history[customer_history["mobile_eft_all_cnt"].isna()].drop("date", axis=1).describe()

In [ ]:
customer_history[customer_history["cust_id"] == 0]

In [ ]:
referance_data[referance_data["cust_id"] == 77]

In [ ]:
customers[customers["cust_id"] == 77]

In [ ]:
customer_history[customer_history["mobile_eft_all_cnt"]==0]

In [ ]:
customer_history[customer_history["cc_transaction_all_cnt"]==0]

In [ ]:
customer_history.loc[customer_history["cust_id"] ==178 ]

In [ ]:
set(customer_history.loc[customer_history["cc_transaction_all_cnt"].isna(), "cust_id"]).intersection(
set(customer_history.loc[customer_history["cc_transaction_all_cnt"].notna(), "cust_id"]))

In [ ]:
history_statics = customer_history.drop("date", axis=1).groupby("cust_id").agg(["min", "max", "median", "mean"])
history_statics.columns = ["_".join(col).strip() for col in history_statics.columns.values]


In [ ]:
history_statics

In [ ]:
customers_with_history = customers.merge(history_statics, "left", on="cust_id")

In [ ]:
train_data = customers_with_history.merge(referance_data, "right", on="cust_id")

In [ ]:
test_data = customers_with_history.merge(referance_data_test, "right", on="cust_id")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [ ]:
churn_counts = train_data['churn'].value_counts()
churn_rate = train_data['churn'].value_counts(normalize=True) * 100

px.pie(values=churn_counts.values,
             names=['Churn Olmayan (0)', 'Churn Olan (1)'],
             title='Müşteri Churn Dağılımı',
             hole=0.3,
             color_discrete_sequence=px.colors.sequential.RdBu)


In [ ]:
numeric_cols = ['age', 'tenure', 'mobile_eft_all_amt_mean', 'cc_transaction_all_amt_mean', 'active_product_category_nbr_mean']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(20, 12))
fig.suptitle('Sayısal Değişkenlerin Dağılımı (Histogram ve Boxplot)', fontsize=16)
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(train_data[col], ax=axes[i], kde=True, color='skyblue')
    axes[i].set_title(f'{col} Dağılımı')

# Son subplot'u boş bırakmamak için gizleyebiliriz
if len(numeric_cols) < len(axes):
    for j in range(len(numeric_cols), len(axes)):
        fig.delaxes(axes[j])

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

In [ ]:
categorical_cols = ['gender', 'work_type', 'work_sector', 'province']

fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(20, 15))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    order = train_data[col].value_counts().index
    sns.countplot(y=train_data[col], ax=axes[i], order=order, palette='viridis')
    axes[i].set_title(f'{col} Dağılımı', fontsize=14)
    axes[i].set_xlabel('Müşteri Sayısı')
    axes[i].set_ylabel('')

plt.tight_layout()
plt.show()

In [ ]:
# Churn oranını hesaplayan bir fonksiyon
def plot_churn_rate_by_category(df, column):
    churn_rate_df = df.groupby(column)['churn'].value_counts(normalize=True).unstack()
    churn_rate_df = churn_rate_df.sort_values(by=1, ascending=False)
    
    fig = px.bar(churn_rate_df,
                 x=churn_rate_df.index,
                 y=1, # Churn olanların oranı
                 title=f'{column.replace("_", " ").title()} Bazında Churn Oranları',
                 labels={'x': column, 'y': 'Churn Oranı (%)'},
                 text=(churn_rate_df[1]*100).apply(lambda x: f'{x:.1f}%'),
                 color_discrete_sequence=['#E57373']) # Kırmızı tonu
    fig.show()

plot_churn_rate_by_category(train_data, 'work_sector')
plot_churn_rate_by_category(train_data, 'province')

In [ ]:
plt.figure(figsize=(18, 15))
# Sadece sayısal ve hedef değişkeni alalım
corr_df = train_data.select_dtypes(include=np.number)
correlation_matrix = corr_df.corr()

sns.heatmap(correlation_matrix, annot=False, cmap='coolwarm')
plt.title('Sayısal Değişkenler Arası Korelasyon Matrisi')
plt.show()

# Churn ile en ilişkili 10 özelliği görelim
print("\nChurn ile en yüksek korelasyona sahip 10 özellik:")
print(correlation_matrix['churn'].abs().sort_values(ascending=False).head(11))

In [ ]:
last_6_months_df_churn  = customer_history[customer_history["cust_id"].isin(referance_data.loc[referance_data["churn"]==1, "cust_id"].tolist())].groupby('cust_id').tail(6).copy()
last_6_months_df_unchurn  = customer_history[customer_history["cust_id"].isin(referance_data.loc[referance_data["churn"]==0, "cust_id"].tolist())].groupby('cust_id').tail(6).copy()

In [ ]:
last_6_months_df_churn

In [ ]:
last_6_months_df_unchurn

In [ ]:
customer_history[customer_history["cust_id"] == 5]

In [ ]:
df = customer_history

In [ ]:
# --- Adım 2: Görselleştirmeler ---

# Grafiklerin daha okunaklı olması için stil belirleyelim
sns.set_theme(style="whitegrid")


# --- Görselleştirme 1: Zaman Serisi - Aylara Göre Toplam İşlem Tutarları ---
print("\n1. Zaman Serisi Grafiği oluşturuluyor...")
monthly_totals = df.groupby('date')[['mobile_eft_all_amt', 'cc_transaction_all_amt']].mean().reset_index()

plt.figure(figsize=(12, 6))
sns.lineplot(data=monthly_totals, x='date', y='mobile_eft_all_amt', label='Mobil EFT Tutarı (€)', marker='o')
sns.lineplot(data=monthly_totals, x='date', y='cc_transaction_all_amt', label='Kredi Kartı Tutarı (€)', marker='o')

plt.title('Aylara Göre Toplam İşlem Tutarlarının Trendi', fontsize=16)
plt.xlabel('Tarih', fontsize=12)
plt.ylabel('Toplam Tutar (€)', fontsize=12)
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


# --- Görselleştirme 2: Karşılaştırma - Toplam Hacim Dağılımı ---
print("2. Karşılaştırma Grafiği (Pasta Grafik) oluşturuluyor...")
total_eft_amt = df['mobile_eft_all_amt'].sum()
total_cc_amt = df['cc_transaction_all_amt'].sum()
labels = ['Mobil EFT', 'Kredi Kartı']
sizes = [total_eft_amt, total_cc_amt]
colors = ['#ff9999','#66b3ff']

plt.figure(figsize=(8, 8))
plt.pie(sizes, labels=labels, autopct='%1.1f%%', startangle=90, colors=colors,
        wedgeprops={'edgecolor': 'black'})
plt.title('Toplam İşlem Hacminin Dağılımı (EFT vs. Kredi Kartı)', fontsize=16)
plt.axis('equal') # Pastayı daire şeklinde gösterir
plt.show()


# --- Görselleştirme 3: Dağılım Analizi - Harcama Tutarlarının Dağılımı ---
print("3. Dağılım Grafiği (Histogram) oluşturuluyor...")
plt.figure(figsize=(12, 6))
sns.histplot(df['cc_transaction_all_amt'], color="skyblue", kde=True, bins=50, label='Kredi Kartı')
sns.histplot(df['mobile_eft_all_amt'], color="red", kde=True, bins=50, label='Mobil EFT')
plt.title('Aylık Harcama Tutarlarının Dağılımı', fontsize=16)
plt.xlabel('İşlem Tutarı (€)', fontsize=12)
plt.ylabel('Müşteri-Ay Sayısı', fontsize=12)
plt.legend()
plt.show()


# --- Görselleştirme 4: İlişki Analizi - İşlem Adedi ve Tutarı İlişkisi ---
print("4. İlişki Grafiği (Saçılım Grafiği) oluşturuluyor...")
# Veri çok büyükse, daha hızlı çizim için rastgele bir örneklem alabiliriz
sample_df = df.sample(n=min(1000, len(df)))

plt.figure(figsize=(10, 6))
sns.scatterplot(data=sample_df, x='cc_transaction_all_cnt', y='cc_transaction_all_amt')
plt.title('Kredi Kartı İşlem Adedi ile Tutarı Arasındaki İlişki', fontsize=16)
plt.xlabel('İşlem Adedi', fontsize=12)
plt.ylabel('Toplam Tutar (€)', fontsize=12)
plt.show()

In [ ]:
customer_history.groupby("active_product_category_nbr").median("cc_transaction_all_cnt")

In [ ]:
customer_history

In [ ]:
customer_history[customer_history["cust_id"].isin(referance_data.loc[referance_data["churn"]==1, "cust_id"])]

In [ ]:
customer_history[customer_history["cust_id"].isin(referance_data.loc[referance_data["churn"]==0, "cust_id"])]

In [ ]:
referance_data

In [ ]:
history_with_churn = customer_history.merge(
    referance_data,"left", 
    left_on=("cust_id", "date"), 
    right_on=("cust_id", "ref_date")
    ).drop("ref_date", axis=1)

In [ ]:
mask = history_with_churn["cust_id"].isin(referance_data["cust_id"])
history_with_churn.loc[mask, "churn"] = history_with_churn.loc[mask, "churn"].fillna(0)

In [ ]:
def fillna_except_last(group): 
    group.iloc[:-1] = group.iloc[:-1].fillna(0)
    return group

history_with_churn["churn"] = history_with_churn.groupby('cust_id')['churn'].transform(fillna_except_last)

In [ ]:
sum(history_with_churn.groupby("cust_id")["churn"].count() == 0)

In [ ]:
referance_data_test.shape

bu şekilde churn null ise test verisidir demiş oluyoruz


In [ ]:
set(referance_data["cust_id"] ).intersection(set(referance_data_test["cust_id"]))

In [ ]:
history_with_churn.isna().sum()

In [ ]:
print(history_with_churn['churn'].value_counts(normalize=True))

In [ ]:
history_with_churn.info()

In [ ]:
df = history_with_churn.copy()

In [ ]:
df.head(19)

In [ ]:
df.pivot_table(columns=["date"], index=["cust_id"], values=["mobile_eft_all_cnt", "mobile_eft_all_amt"])

In [ ]:
import psutil
import os
import pickle
import pandas as pd
from tqdm import tqdm
import gc


In [ ]:

def pivot_windows_ultra_safe(input_file, output_file='pivoted_data.parquet',
                             checkpoint_file='pivot_checkpoint.pkl',
                             chunk_size=1000, window_size=19):
    """
    Checkpoint sistemi + bellek izleme
    """
    feature_cols = ['mobile_eft_all_cnt', 'active_product_category_nbr', 
                    'mobile_eft_all_amt', 'cc_transaction_all_amt', 
                    'cc_transaction_all_cnt', 'churn']
    
    # Checkpoint varsa yükle
    processed_windows = set()
    first_write = True
    
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'rb') as f:
            processed_windows = pickle.load(f)
        first_write = False
        print(f"Checkpoint yüklendi: {len(processed_windows)} window işlendi")
    
    # Müşteri ID'lerini oku
    customer_ids = pd.read_parquet(input_file, columns=['cust_id'])['cust_id'].unique()
    
    temp_batch = []
    total_windows = 0
    
    for idx, cust_id in enumerate((customer_ids)):
        # Bellek kontrolü
        memory_percent = psutil.virtual_memory().percent
        if memory_percent > 85:
            print(f"\n⚠️ Bellek kullanımı yüksek ({memory_percent}%), temizlik yapılıyor...")
            gc.collect()
            
        try:
            # Müşterinin verisini oku
            customer_df = pd.read_parquet(
                input_file,
                filters=[('cust_id', '==', cust_id)]
            )
            
            customer_df = customer_df.sort_values(['window_id', 'date'])
            
            for window_id in customer_df['window_id'].unique():
                # Zaten işlendiyse atla
                if (cust_id, window_id) in processed_windows:
                    continue
                
                window_data = customer_df[customer_df['window_id'] == window_id]
                
                if len(window_data) != window_size:
                    continue
                
                pivot_row = {'cust_id': cust_id, 'window_id': window_id}
                
                for month_idx in range(1, window_size + 1):
                    month_data = window_data.iloc[month_idx - 1]
                    for col in feature_cols:
                        pivot_row[f'Month{month_idx}_{col}'] = month_data[col]
                
                temp_batch.append(pivot_row)
                processed_windows.add((cust_id, window_id))
                total_windows += 1
                print(len(temp_batch))
                # Batch dolduğunda yaz
                if len(temp_batch) >= chunk_size:
                    chunk_df = pd.DataFrame(temp_batch)
                    
                    if first_write:
                        chunk_df.to_parquet(output_file, index=False, engine='fastparquet')
                        first_write = False
                    else:
                        chunk_df.to_parquet(output_file, index=False, engine='fastparquet', 
                                           append=True)
                    
                    temp_batch = []
                    gc.collect()
            
            del customer_df
            
            # Her 50 müşteride checkpoint kaydet
            if (idx + 1) % 50 == 0:
                with open(checkpoint_file, 'wb') as f:
                    pickle.dump(processed_windows, f)
                gc.collect()
        
        except Exception as e:
            print(f"\n❌ Hata (cust_id={cust_id}): {e}")
            # Checkpoint kaydet
            with open(checkpoint_file, 'wb') as f:
                pickle.dump(processed_windows, f)
            raise
    
    # Kalan batch
    if temp_batch:
        chunk_df = pd.DataFrame(temp_batch)
        if first_write:
            chunk_df.to_parquet(output_file, index=False, engine='fastparquet')
        else:
            chunk_df.to_parquet(output_file, index=False, engine='fastparquet', 
                               append=True)
    
    # Checkpoint temizle
    if os.path.exists(checkpoint_file):
        os.remove(checkpoint_file)
    
    print(f"\nToplam {total_windows} window işlendi")
    print(f"Veri {output_file} dosyasına yazıldı")
    return output_file

# Kullanım
pivoted_file = pivot_windows_ultra_safe('windowed_data.parquet',
                                        output_file='pivoted_data.parquet',
                                        chunk_size=500)

In [ ]:
windowed_data = pd.read_parquet("windowed_data.parquet")

In [ ]:
feature_cols = ['mobile_eft_all_cnt', 'active_product_category_nbr', 
                    'mobile_eft_all_amt', 'cc_transaction_all_amt', 
                    'cc_transaction_all_cnt', 'churn']
windowed_data["new_date"] = (windowed_data.index % 19).map(lambda x: f"month{x}")
#windowed_data.pivot_table(index=["cust_id", "window_id"], columns=["new_date"], values=feature_cols)

In [ ]:
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
import os

feature_cols = [
    'mobile_eft_all_cnt', 'active_product_category_nbr', 
    'mobile_eft_all_amt', 'cc_transaction_all_amt', 
    'cc_transaction_all_cnt', 'churn'
]
group_cols = ['cust_id', 'window_id']
all_needed_cols = group_cols + feature_cols

# Girdi ve Çıktı dosyaları
input_path = "windowed_data.parquet"
output_path = "pivoted_data.parquet"

# --- Adım 1: Belleğe sığacak şekilde sadece gruplama anahtarlarını oku ---
print("İşlenecek benzersiz müşteri grupları belirleniyor...")
# Sadece gruplama yapılacak sütunları okuyarak benzersiz grupları buluyoruz.
# Bu işlem genellikle tüm veriyi okumaktan çok daha az bellek kullanır.
unique_groups = pd.read_parquet(input_path, columns=group_cols).drop_duplicates()
print(f"Toplam {len(unique_groups):,} benzersiz grup bulundu.")

# Çıkış dosyası varsa sil
if os.path.exists(output_path):
    os.remove(output_path)

writer = None
processed_groups = 0

try:
    # --- Adım 2: Bu grupları daha küçük parçalara ayırarak işle ---
    group_chunk_size = 500  # Her döngüde kaç müşteri grubunu işleyeceğimizi belirler. RAM'e göre ayarlayın.
    
    for i in range(0, len(unique_groups), group_chunk_size):
        # İşlenecek müşteri grubu grubunu seç
        group_chunk = unique_groups.iloc[i:i+group_chunk_size]
        
        # --- Adım 3: Sadece o gruplara ait veriyi dosyadan filtreleyerek oku ---
        # PyArrow'un filtreleme özelliği sayesinde tüm dosyayı belleğe almadan
        # sadece istediğimiz satırları verimli bir şekilde okuyabiliriz.
        filters = [
            ('cust_id', 'in', group_chunk['cust_id'].tolist()),
            ('window_id', 'in', group_chunk['window_id'].tolist())
        ]
        
        # Filtrelenmiş veriyi Pandas DataFrame olarak oku
        # Not: Burada sadece ihtiyaç duyulan sütunları okuyarak daha da verimli hale getiriyoruz.
        data_chunk = pq.read_table(source=input_path, columns=all_needed_cols, filters=filters).to_pandas()
        
        # --- Adım 4: Okunan küçük parça üzerinde işlemleri yap ---
        data_chunk["new_date"] = (data_chunk.index % 19).map(lambda x: f"month{x}")
        
        pivot_chunk = data_chunk.pivot_table(
            index=group_cols, 
            columns=["new_date"], 
            values=feature_cols
        )
        
        pivot_chunk.columns = [f"{col[0]}_{col[1]}" for col in pivot_chunk.columns.to_flat_index()]
        pivot_chunk.reset_index(inplace=True)
        
        # --- Adım 5: İşlenen parçayı sonuç dosyasına ekle ---
        table = pa.Table.from_pandas(pivot_chunk)
        
        if writer is None:
            writer = pq.ParquetWriter(output_path, table.schema, use_dictionary=True)
        
        writer.write_table(table)
        
        processed_groups += len(group_chunk)
        print(f"Processed {processed_groups:,}/{len(unique_groups):,} groups")

finally:
    if writer:
        writer.close()
        print(f"\n✅ Pivot sonuçları '{output_path}' dosyasına başarıyla yazıldı.")


In [ ]:
pd.read_parquet("pivoted_windowed_data_memory_safe.parquet")